# Engineering Drawing Comment Extractor (Colab)

Processes **every PDF in the `pdfs` folder**, page by page:

1. **Native annotations first** – PyMuPDF reads real PDF comments (text, author, date, colour, box, callout arrow).
2. **No annotations on the page?** → the page is rasterised and sent to **LightOnOCR-2-1B (loaded locally on the Colab GPU)**.
3. Fuzzy de-duplication across the document → *repeated*.
4. Hosted-LLM classification (OpenRouter) → *technical / aesthetic*.
5. Screenshot crop for each comment.
6. Outputs per PDF: `comments.json`, `final_output.csv`, `summary.csv`, `images/` + a combined `ALL_comments.csv`.

**Before running:** Runtime → Change runtime type → **T4 GPU** (OCR on CPU works but is very slow).
Add `OPENROUTER_API_KEY` (and optionally `HF_TOKEN`) in Colab's 🔑 **Secrets** panel, or just paste them when prompted.

## 1. Install dependencies

In [ ]:
!pip install -q pymupdf rapidfuzz openai pyyaml pandas tqdm pillow accelerate huggingface_hub
!pip install -q -U transformers

# LightOnOCR-2 needs a transformers version that has the LightOnOcr classes.
import subprocess, sys
_probe = subprocess.run(
    [sys.executable, "-c", "from transformers import LightOnOcrForConditionalGeneration, LightOnOcrProcessor"],
    capture_output=True, text=True)
if _probe.returncode != 0:
    print("Stable transformers lacks LightOnOcr classes -> installing from source (takes ~1-2 min)...")
    subprocess.run([sys.executable, "-m", "pip", "install", "-q", "git+https://github.com/huggingface/transformers"], check=True)
else:
    print("transformers OK")

## 2. API keys and PDF folder

In [ ]:
import os
from getpass import getpass
from pathlib import Path

def get_secret(name: str, required: bool = True):
    """Order: env var -> Colab Secrets -> interactive prompt."""
    val = os.getenv(name)
    if not val:
        try:
            from google.colab import userdata
            val = userdata.get(name)
        except Exception:
            val = None
    if not val and required:
        val = getpass(f"Enter {name}: ")
    if val:
        os.environ[name] = val
    return val

get_secret("OPENROUTER_API_KEY", required=True)   # metadata + classification LLM
get_secret("HF_TOKEN", required=False)            # optional: faster / authenticated model download

# ---- where are the PDFs? -------------------------------------------------
USE_GOOGLE_DRIVE = False                          # True -> read PDFs from / write outputs to Drive
DRIVE_BASE = "/content/drive/MyDrive/comment_extractor"   # expects  <DRIVE_BASE>/pdfs/*.pdf

if USE_GOOGLE_DRIVE:
    from google.colab import drive
    drive.mount("/content/drive")
    BASE_DIR = Path(DRIVE_BASE)
else:
    BASE_DIR = Path("/content")

PDF_DIR = BASE_DIR / "pdfs"
OUTPUT_DIR = BASE_DIR / "output"
PDF_DIR.mkdir(parents=True, exist_ok=True)
OUTPUT_DIR.mkdir(parents=True, exist_ok=True)

# If the pdfs folder is empty, let the user upload files straight into it.
if not list(PDF_DIR.glob("**/*.pdf")):
    print(f"No PDFs found in {PDF_DIR}. Upload them now:")
    from google.colab import files
    for name, data in files.upload().items():
        (PDF_DIR / name).write_bytes(data)

print("PDFs found:", len(list(PDF_DIR.glob("**/*.pdf"))), "in", PDF_DIR)

## 3. Configuration
Edit this cell if you want to change models, batch sizes, etc.

In [ ]:
%%writefile config.yaml
# LLM Configuration (OpenRouter) - used for title-block metadata + comment classification
llm:
  base_url: "https://openrouter.ai/api/v1"
  model: "stealth/space-bunny-alpha"
  api_key_env: "OPENROUTER_API_KEY"
  batch_size: 40                       # max comments per classification call
  max_workers: 4                       # parallel classification calls (I/O bound)
  timeout: 90
  max_retries: 3

# OCR fallback: runs LOCALLY on the Colab GPU, only for pages with no native annotations
ocr:
  enabled: true
  model: "lightonai/LightOnOCR-2-1B"
  api_key_env: "HF_TOKEN"              # optional, only used for downloading weights
  dpi: 150                             # DPI used to rasterise the page for OCR
  max_new_tokens: 2048                 # max OCR output length per page
  llm_filter: true                     # LightOnOCR transcribes the WHOLE page; let the LLM keep only review comments

vision:
  enabled: false
  model: ""

title_block:
  # [x0, y0, x1, y1] as fractions of the page (0,0 = top-left). Default: bottom-right corner.
  region: [0.5, 0.7, 1.0, 1.0]

classification:
  similarity_threshold: 90             # rapidfuzz ratio for "repeated"

screenshot:
  margin: 1.15                         # crop = comment box (+ arrow) scaled by this factor
  min_width: 200                       # points
  min_height: 150                      # points
  dpi: 150

uid:
  row_density: 80

## 4. Core code

In [ ]:
import base64
import json
import logging
import re
import tempfile
import time
from concurrent.futures import ThreadPoolExecutor
from dataclasses import dataclass
from typing import Dict, List, Optional, Tuple

import fitz  # PyMuPDF
import pandas as pd
import yaml
from openai import OpenAI
from PIL import Image
from rapidfuzz import fuzz, process
from tqdm.auto import tqdm

logging.basicConfig(level=logging.INFO, format="%(asctime)s | %(levelname)s | %(message)s", force=True)
logger = logging.getLogger("comment_extractor")


@dataclass
class Config:
    llm_base_url: str
    llm_model: str
    llm_api_key_env: str
    llm_batch_size: int
    llm_max_workers: int
    llm_timeout: int
    llm_max_retries: int
    title_block_region: Tuple[float, float, float, float]
    similarity_threshold: int
    screenshot_margin: float
    screenshot_min_width: int
    screenshot_min_height: int
    screenshot_dpi: int
    row_density: int
    vision_enabled: bool = False
    vision_model: str = ""
    # OCR fallback (local LightOnOCR-2-1B)
    ocr_enabled: bool = True
    ocr_model: str = "lightonai/LightOnOCR-2-1B"
    ocr_api_key_env: str = "HF_TOKEN"
    ocr_dpi: int = 150
    ocr_max_new_tokens: int = 2048
    ocr_llm_filter: bool = True

    @classmethod
    def from_yaml(cls, path: Path) -> "Config":
        with open(path, encoding="utf-8") as f:
            y = yaml.safe_load(f)
        llm = y["llm"]
        shot = y.get("screenshot", {})
        vision = y.get("vision", {})
        ocr = y.get("ocr", {})
        return cls(
            llm_base_url=llm["base_url"],
            llm_model=llm["model"],
            llm_api_key_env=llm.get("api_key_env", "LLM_API_KEY"),
            llm_batch_size=llm.get("batch_size", 40),
            llm_max_workers=llm.get("max_workers", 4),
            llm_timeout=llm.get("timeout", 90),
            llm_max_retries=llm.get("max_retries", 3),
            title_block_region=tuple(y.get("title_block", {}).get("region", [0.5, 0.7, 1.0, 1.0])),
            similarity_threshold=y.get("classification", {}).get("similarity_threshold", 90),
            screenshot_margin=shot.get("margin", 1.15),
            screenshot_min_width=shot.get("min_width", 200),
            screenshot_min_height=shot.get("min_height", 150),
            screenshot_dpi=shot.get("dpi", 150),
            row_density=y.get("uid", {}).get("row_density", 80),
            vision_enabled=vision.get("enabled", False),
            vision_model=vision.get("model", "") or llm["model"],
            ocr_enabled=ocr.get("enabled", True),
            ocr_model=ocr.get("model", "lightonai/LightOnOCR-2-1B"),
            ocr_api_key_env=ocr.get("api_key_env", "HF_TOKEN"),
            ocr_dpi=ocr.get("dpi", 150),
            ocr_max_new_tokens=ocr.get("max_new_tokens", 2048),
            ocr_llm_filter=ocr.get("llm_filter", True),
        )


def parse_pdf_date(date_str: str) -> Tuple[str, str]:
    """PDF date 'D:YYYYMMDDHHmmSS...' -> ('DD/MM/YYYY', 'HH:MM:SS') or ('NA','NA')."""
    if not date_str:
        return "NA", "NA"
    s = date_str.strip()
    if s.startswith("D:"):
        s = s[2:]
    digits = re.sub(r"[^0-9]", "", s)
    if len(digits) < 8:
        return "NA", "NA"
    hour = digits[8:10] if len(digits) >= 10 else "00"
    mins = digits[10:12] if len(digits) >= 12 else "00"
    secs = digits[12:14] if len(digits) >= 14 else "00"
    return f"{digits[6:8]}/{digits[4:6]}/{digits[0:4]}", f"{hour}:{mins}:{secs}"


def detect_annotation_color(annot) -> str:
    """Human-readable name of the annotation's stroke/fill colour."""
    try:
        colors = annot.colors or {}
        rgb = colors.get("stroke") or colors.get("fill")
        if not rgb or len(rgb) < 3:
            return "Unknown"
        r, g, b = float(rgb[0]), float(rgb[1]), float(rgb[2])
        if r > 0.7 and g < 0.35 and b < 0.35:
            return "Red"
        if b > 0.7 and r < 0.35 and g < 0.5:
            return "Blue"
        if g > 0.7 and r < 0.4 and b < 0.4:
            return "Green"
        if r > 0.8 and g > 0.6 and b < 0.3:
            return "Yellow"
        if r < 0.2 and g < 0.2 and b < 0.2:
            return "Black"
        if r > 0.9 and g > 0.9 and b > 0.9:
            return "White"
        if r > 0.7 and b > 0.7 and g < 0.3:
            return "Magenta"
        return f"RGB({round(r*255)},{round(g*255)},{round(b*255)})"
    except Exception:
        return "Unknown"


def extract_json(text: str):
    """Pull the first JSON object/array out of an LLM reply (tolerates fences and <think> blocks)."""
    text = re.sub(r"<think>.*?</think>", "", text, flags=re.S).strip()
    text = re.sub(r"^```(?:json)?|```$", "", text, flags=re.M).strip()
    starts = [i for i in (text.find("{"), text.find("[")) if i != -1]
    if not starts:
        raise ValueError("No JSON found in model reply")
    obj, _ = json.JSONDecoder().raw_decode(text[min(starts):])
    return obj

### Hosted LLM client (OpenRouter)

In [ ]:
class LLMClient:
    def __init__(self, cfg: Config):
        self.cfg = cfg
        key = os.getenv(cfg.llm_api_key_env)
        if not key:
            raise EnvironmentError(f"{cfg.llm_api_key_env} is not set (run the API-keys cell first).")
        self.client = OpenAI(base_url=cfg.llm_base_url, api_key=key, timeout=cfg.llm_timeout)
        self.model = cfg.llm_model
        self.max_retries = cfg.llm_max_retries

    def _content(self, resp) -> str:
        """Return the reply text, or raise with OpenRouter's real error message."""
        if not getattr(resp, "choices", None):
            err = getattr(resp, "error", None) or (getattr(resp, "model_extra", None) or {}).get("error") or resp
            raise RuntimeError(f"Provider returned no choices for model '{self.model}': {err}")
        content = resp.choices[0].message.content
        if not content:
            raise RuntimeError(f"Model '{self.model}' returned empty content "
                               f"(finish_reason={resp.choices[0].finish_reason})")
        return content

    def chat_json(self, system: str, user: str):
        last: Optional[Exception] = None
        for attempt in range(self.max_retries):
            try:
                resp = self.client.chat.completions.create(
                    model=self.model,
                    messages=[
                        {"role": "system", "content": system},
                        {"role": "user", "content": user},
                    ],
                    temperature=0,
                    max_tokens=4000,
                )
                return extract_json(self._content(resp))
            except Exception as e:  # network, rate limit, bad JSON ...
                last = e
                wait = 2 ** attempt
                logger.warning(f"LLM call failed ({e}); retry {attempt + 1}/{self.max_retries} in {wait}s")
                time.sleep(wait)
        raise last  # type: ignore[misc]

    def inspect_snapshot_vision(self, image_path: Path, comment_text: str, annot_type: str) -> Optional[Dict]:
        if not image_path.exists():
            return None
        with open(image_path, "rb") as f:
            b64_img = base64.b64encode(f.read()).decode("utf-8")

        prompt = (
            f"You are inspecting a cropped snippet from an engineering drawing containing an annotation.\n"
            f"Annotation Type: {annot_type}\n"
            f"Annotation Note: '{comment_text}'\n\n"
            "Analyze the image and return a JSON object with exactly these fields:\n"
            '- "is_handwritten": "Y" if any handwritten note or handwritten mark is present, otherwise "N"\n'
            '- "visual_feature": concise description of the engineering equipment, line, or area being marked (e.g. pump P-101, valve, nozzle, pipe line)\n'
            '- "category": "technical" if related to engineering/dimensions/equipment, or "aesthetic" if cosmetic/formatting\n'
            '- "confidence_percent": integer 0-100\n\n'
            'Reply ONLY with JSON: {"is_handwritten": "N", "visual_feature": "...", "category": "technical", "confidence_percent": 95}'
        )
        for attempt in range(self.max_retries):
            try:
                resp = self.client.chat.completions.create(
                    model=self.cfg.vision_model or self.model,
                    messages=[{
                        "role": "user",
                        "content": [
                            {"type": "text", "text": prompt},
                            {"type": "image_url", "image_url": {"url": f"data:image/png;base64,{b64_img}"}},
                        ],
                    }],
                    temperature=0,
                    max_tokens=1000,
                )
                return extract_json(self._content(resp))
            except Exception as e:
                wait = 2 ** attempt
                logger.warning(f"Vision API attempt {attempt + 1} failed: {e}")
                time.sleep(wait)
        return None

### OCR fallback – LightOnOCR-2-1B running locally
The model is loaded **lazily** the first time a page with no native annotations is hit, so PDFs that are fully annotated never pay the load cost.

> LightOnOCR is a pure transcription model (it reads the whole page and ignores "only extract comments" instructions),
> so the raw transcript is cleaned and then, if `ocr.llm_filter: true`, the hosted LLM keeps only the lines that look like review comments
> (and drops tags, titles, dimensions, drawing numbers).

In [ ]:
class OCREngine:
    """Local LightOnOCR-2-1B. Weights are downloaded and loaded on first use."""

    def __init__(self, cfg: Config):
        self.cfg = cfg
        self.model = None
        self.processor = None
        self.device = None
        self.dtype = None

    # ---- model loading -------------------------------------------------
    def _load(self):
        if self.model is not None:
            return
        import torch
        from transformers import LightOnOcrForConditionalGeneration, LightOnOcrProcessor

        self.device = "cuda" if torch.cuda.is_available() else "cpu"
        if self.device == "cuda":
            self.dtype = torch.bfloat16 if torch.cuda.is_bf16_supported() else torch.float16
        else:
            self.dtype = torch.float32
            logger.warning("No GPU detected - OCR will be slow. Runtime > Change runtime type > T4 GPU.")

        token = os.getenv(self.cfg.ocr_api_key_env) or None
        logger.info(f"Loading {self.cfg.ocr_model} on {self.device} ({self.dtype}) ...")
        try:
            model = LightOnOcrForConditionalGeneration.from_pretrained(
                self.cfg.ocr_model, dtype=self.dtype, token=token)
        except TypeError:  # older transformers builds use torch_dtype
            model = LightOnOcrForConditionalGeneration.from_pretrained(
                self.cfg.ocr_model, torch_dtype=self.dtype, token=token)
        self.model = model.to(self.device).eval()
        self.processor = LightOnOcrProcessor.from_pretrained(self.cfg.ocr_model, token=token)
        logger.info("OCR model ready.")

    # ---- single page ---------------------------------------------------
    def _run_image(self, img: Image.Image) -> str:
        import torch

        tmp = tempfile.NamedTemporaryFile(suffix=".png", delete=False)
        tmp.close()
        img.save(tmp.name)
        try:
            messages = [{"role": "user", "content": [{"type": "image", "url": tmp.name}]}]
            inputs = self.processor.apply_chat_template(
                messages, add_generation_prompt=True, tokenize=True,
                return_dict=True, return_tensors="pt")
            inputs = {
                k: (v.to(self.device, dtype=self.dtype) if v.is_floating_point() else v.to(self.device))
                for k, v in inputs.items()
            }
            with torch.inference_mode():
                out_ids = self.model.generate(**inputs, max_new_tokens=self.cfg.ocr_max_new_tokens)
            gen = out_ids[0, inputs["input_ids"].shape[1]:]
            text = self.processor.decode(gen, skip_special_tokens=True)
        finally:
            os.unlink(tmp.name)
            if self.device == "cuda":
                torch.cuda.empty_cache()
        return text

    def ocr_page(self, page: fitz.Page) -> List[str]:
        """Rasterise *page* at cfg.ocr_dpi, OCR it, return cleaned non-empty text lines."""
        self._load()
        zoom = self.cfg.ocr_dpi / 72.0
        pix = page.get_pixmap(matrix=fitz.Matrix(zoom, zoom), annots=True, alpha=False)
        img = Image.frombytes("RGB", (pix.width, pix.height), pix.samples)

        try:
            raw_text = self._run_image(img)
        except Exception as exc:
            logger.error(f"OCR failed on page: {exc}")
            raise RuntimeError(f"OCR failed: {exc}") from exc

        # strip HTML / markdown noise
        text_no_html = re.sub(r"<[^>]*>", "", raw_text, flags=re.DOTALL)
        segments: List[str] = []
        for line in text_no_html.splitlines():
            clean = re.sub(r"^#+\s*", "", line).strip()
            if clean in ("---", "***", "___", "--", "-", "```", "```plaintext", "plaintext"):
                continue
            if clean.startswith("```") or clean.startswith("<") or "style=" in clean \
                    or "border:" in clean or "display:" in clean or "justify-content:" in clean:
                continue
            if len(clean) >= 3:
                segments.append(clean)
        return segments


FILTER_SYSTEM = "You filter OCR output from engineering drawings. Reply with JSON only."


def filter_ocr_segments(llm: LLMClient, segments: List[str], chunk: int = 60) -> List[str]:
    """Keep only segments that look like review comments (instruction / change request / observation)."""
    kept: List[str] = []
    for k in range(0, len(segments), chunk):
        part = segments[k:k + chunk]
        payload = [{"id": i, "text": t} for i, t in enumerate(part)]
        user = (
            "Below are text lines OCR'd from ONE page of an engineering drawing.\n"
            "Keep ONLY lines that are review comments: an instruction, change, request, observation or check "
            "(e.g. \"Increase diameter to 70mm\", \"Move pump 200mm\", \"Change valve to 4in\"). "
            "They may contain numbers, units or symbols.\n"
            "Drop: equipment tags / IDs / part numbers / drawing numbers (e.g. MS045I, V-102, PT-102), "
            "titles, labels, legends, plain dimensions, title-block text and ordinary drawing text.\n\n"
            'Return JSON only: {"keep": [<ids of lines to keep>]}\n\n'
            f"Lines:\n{json.dumps(payload, ensure_ascii=False)}"
        )
        try:
            data = llm.chat_json(FILTER_SYSTEM, user)
            ids = data.get("keep", []) if isinstance(data, dict) else data
            kept.extend(part[int(i)] for i in ids if 0 <= int(i) < len(part))
        except Exception as e:
            logger.warning(f"OCR comment filter failed ({e}); keeping all lines of this chunk")
            kept.extend(part)
    return kept


def ocr_page_to_comments(engine: OCREngine, page: fitz.Page, page_no: int,
                         llm: Optional[LLMClient] = None) -> List[Dict]:
    """OCR a page and convert text lines into comment dicts (same schema as collect_comments)."""
    segments = engine.ocr_page(page)
    n_raw = len(segments)
    if segments and llm is not None and engine.cfg.ocr_llm_filter:
        segments = filter_ocr_segments(llm, segments)
    if not segments:
        logger.info(f"Page {page_no}: OCR read {n_raw} line(s), 0 kept as comments")
        return []

    out: List[Dict] = []
    pr = page.rect  # whole page is used as the "box" for OCR-sourced items
    for seg in segments:
        out.append({
            "page": page_no,
            "text": re.sub(r"\s+", " ", seg),
            "author": "OCR",
            "date": "NA",
            "time": "NA",
            "color": "Unknown",
            "rect": fitz.Rect(pr),
            "arrow_pts": [],
            "annot_type": "OCR",
        })
    logger.info(f"Page {page_no}: OCR read {n_raw} line(s), {len(out)} kept as comments")
    return out

### Metadata, native annotation collection, classification

In [ ]:
# ============================ METADATA ============================

def parse_filename(filename: str) -> Dict:
    """Expected: PROJECT_SOMETHING_DRAWINGNUMBER_REVISION.pdf"""
    parts = Path(filename).stem.split("_")
    return {
        "project": parts[0] if len(parts) > 0 else "UNKNOWN",
        "drawing_number": parts[2] if len(parts) > 2 else "UNKNOWN",
        "praj_revision": parts[3] if len(parts) > 3 else "R0",
        "customer_doc": "NA",
        "customer_revision": "NA",
    }


def extract_metadata(doc: fitz.Document, filename: str, llm: LLMClient, cfg: Config) -> Dict:
    """Filename first, then override with whatever the title block (text layer) yields."""
    meta = parse_filename(filename)
    if len(doc) == 0:
        return meta

    page = doc[0]
    r = page.rect
    x0, y0, x1, y1 = cfg.title_block_region
    clip = fitz.Rect(
        r.x0 + r.width * x0, r.y0 + r.height * y0,
        r.x0 + r.width * x1, r.y0 + r.height * y1,
    )
    tb_text = page.get_text("text", clip=clip).strip()[:4000]
    if not tb_text:
        logger.info("Title block has no text layer; using filename metadata only")
        return meta

    system = "You extract fields from engineering drawing title blocks. Reply with JSON only."
    user = (
        "Below is raw text from the title block of an engineering drawing.\n"
        "Return a JSON object with exactly these keys (use null if a value is not present):\n"
        '  "project"                  - project name or code\n'
        '  "drawing_number"           - the issuing company\'s own drawing/document number\n'
        '  "revision"                 - the issuing company\'s revision of this drawing\n'
        '  "customer_document_number" - the customer\'s / client\'s document number\n'
        '  "customer_revision"        - the customer\'s revision\n'
        "Do not guess. Copy values exactly as written.\n\n"
        f"Title block text:\n{tb_text}"
    )
    try:
        data = llm.chat_json(system, user)
        if isinstance(data, dict):
            def clean(v):
                return str(v).strip() if v not in (None, "", "null") else None

            for key, field in [
                ("project", "project"),
                ("drawing_number", "drawing_number"),
                ("revision", "praj_revision"),
                ("customer_document_number", "customer_doc"),
                ("customer_revision", "customer_revision"),
            ]:
                v = clean(data.get(key))
                if v:
                    meta[field] = v
    except Exception as e:
        logger.warning(f"Title-block extraction failed, keeping filename metadata: {e}")
    return meta


# ============================ NATIVE ANNOTATIONS ============================

SKIP_TYPES = {"Link", "Widget", "Popup"}


def callout_points(doc: fitz.Document, page: fitz.Page, annot) -> List[fitz.Point]:
    """Callout-line vertices (/CL) of a FreeText callout, in page coordinates (best effort)."""
    try:
        kind, val = doc.xref_get_key(annot.xref, "CL")
        if kind != "array":
            return []
        nums = [float(n) for n in re.findall(r"-?\d+(?:\.\d+)?", val)]
        m = page.transformation_matrix * page.rotation_matrix
        pts = [fitz.Point(nums[i], nums[i + 1]) * m for i in range(0, len(nums) - 1, 2)]
        bounds = page.rect + (-50, -50, 50, 50)
        return [p for p in pts if bounds.contains(p)]
    except Exception:
        return []


def collect_comments(doc: fitz.Document, page: fitz.Page, page_no: int) -> List[Dict]:
    out: List[Dict] = []
    for annot in page.annots() or []:
        atype = annot.type[1]
        if atype in SKIP_TYPES:
            continue

        info = annot.info
        text = (info.get("content") or "").strip()

        # Some FreeText boxes keep their text only in the appearance stream.
        if not text and atype == "FreeText":
            text = page.get_text("text", clip=annot.rect).strip()
        if not text:
            continue

        date, time_val = parse_pdf_date(info.get("creationDate") or info.get("modDate") or "")
        out.append({
            "page": page_no,
            "text": re.sub(r"\s+", " ", text),
            "author": (info.get("title") or "").strip() or "Unknown",
            "date": date,
            "time": time_val,
            "color": detect_annotation_color(annot),
            "rect": fitz.Rect(annot.rect),
            "arrow_pts": callout_points(doc, page, annot),
            "annot_type": atype,
        })
    return out


# ============================ CLASSIFICATION ============================

VALID_CATEGORIES = {"technical", "aesthetic"}

CLASSIFY_SYSTEM = "You classify review comments written on engineering drawings. Reply with JSON only."

AESTHETIC_KW = ["font", "colour", "color", "label", "format", "align", "spacing", "typo",
                "text size", "presentation", "layout", "heading", "style", "border",
                "margin", "indent"]
TECHNICAL_KW = ["dimension", "material", "tolerance", "weld", "bolt", "pressure",
                "temperature", "load", "stress", "code", "standard", "spec", "shall",
                "must", "remark", "note", "verify", "check", "revise", "incorrect",
                "missing"]


def rule_based_category(text: str) -> str:
    t = text.lower()
    a = sum(k in t for k in AESTHETIC_KW)
    b = sum(k in t for k in TECHNICAL_KW)
    return "aesthetic" if a > b else "technical"


def classify_batch(llm: LLMClient, batch: List[Tuple[int, str]]) -> Dict[int, Tuple[str, str]]:
    """batch = [(comment_index, text)] -> {comment_index: (category, method)}"""
    payload = [{"id": i, "text": t} for i, t in batch]
    user = (
        "Classify each engineering drawing review comment as exactly one of:\n"
        "- technical: dimensions, materials, tolerances, code/standard compliance, process, "
        "structural or design issues, missing or incorrect data\n"
        "- aesthetic: formatting, labels, fonts, colours, typography, alignment, "
        "visual presentation\n\n"
        'Return JSON only: {"results": [{"id": <id>, "category": "technical" or "aesthetic"}]}\n'
        "Include every id exactly once.\n\n"
        f"Comments:\n{json.dumps(payload, ensure_ascii=False)}"
    )
    out: Dict[int, Tuple[str, str]] = {}
    try:
        data = llm.chat_json(CLASSIFY_SYSTEM, user)
        rows = data.get("results", []) if isinstance(data, dict) else data
        for row in rows:
            cat = str(row.get("category", "")).lower().strip()
            if cat in VALID_CATEGORIES:
                out[int(row["id"])] = (cat, "llm")
    except Exception as e:
        logger.warning(f"Classification call failed, using rule-based fallback: {e}")

    for i, text in batch:  # anything missing or invalid -> rules
        if i not in out:
            out[i] = (rule_based_category(text), "rule")
    return out


def classify_document(comments: List[Dict], llm: LLMClient, cfg: Config) -> None:
    """Fills comment['category'] and comment['method'] in place."""
    # Stage 1: repeats across the whole document (first occurrence is kept as the original)
    seen: List[str] = []
    pending: List[int] = []
    for idx, c in enumerate(comments):
        norm = c["text"].lower().strip()
        if seen and process.extractOne(norm, seen, scorer=fuzz.ratio, score_cutoff=cfg.similarity_threshold):
            c["category"], c["method"] = "repeated", "similarity"
        else:
            seen.append(norm)
            pending.append(idx)

    # Stage 2: LLM, page by page, chunked
    by_page: Dict[int, List[int]] = {}
    for idx in pending:
        by_page.setdefault(comments[idx]["page"], []).append(idx)

    batches: List[List[Tuple[int, str]]] = []
    for page_no in sorted(by_page):
        idxs = by_page[page_no]
        for k in range(0, len(idxs), cfg.llm_batch_size):
            batches.append([(i, comments[i]["text"]) for i in idxs[k:k + cfg.llm_batch_size]])

    with ThreadPoolExecutor(max_workers=cfg.llm_max_workers) as ex:
        for result in ex.map(lambda b: classify_batch(llm, b), batches):
            for i, (cat, method) in result.items():
                comments[i]["category"], comments[i]["method"] = cat, method

### Screenshots, UID and the per-PDF pipeline (annotations first → OCR only if the page is empty)

In [ ]:
def capture_screenshot(page: fitz.Page, c: Dict, uid: str, img_dir: Path, cfg: Config) -> str:
    """Crop around the comment box and its arrow target; grow to min size; stay inside the page."""
    box = fitz.Rect(c["rect"])
    for p in c["arrow_pts"]:
        box.include_point(p)

    cx, cy = (box.x0 + box.x1) / 2, (box.y0 + box.y1) / 2
    w = max(box.width * cfg.screenshot_margin, cfg.screenshot_min_width)
    h = max(box.height * cfg.screenshot_margin, cfg.screenshot_min_height)
    crop = fitz.Rect(cx - w / 2, cy - h / 2, cx + w / 2, cy + h / 2)

    pr = page.rect  # shift the crop back inside the page rather than shrinking it
    if crop.x0 < pr.x0:
        crop.x1 += pr.x0 - crop.x0
        crop.x0 = pr.x0
    if crop.x1 > pr.x1:
        crop.x0 -= crop.x1 - pr.x1
        crop.x1 = pr.x1
    if crop.y0 < pr.y0:
        crop.y1 += pr.y0 - crop.y0
        crop.y0 = pr.y0
    if crop.y1 > pr.y1:
        crop.y0 -= crop.y1 - pr.y1
        crop.y1 = pr.y1
    crop = crop & pr

    zoom = cfg.screenshot_dpi / 72.0
    pix = page.get_pixmap(clip=crop, matrix=fitz.Matrix(zoom, zoom), annots=True)
    out = img_dir / f"{uid}.png"
    pix.save(str(out))
    return str(out).replace("\\", "/")


def make_uid(meta: Dict, page_no: int, y_pos: float, page_height: float, seq: int, cfg: Config) -> str:
    row = min(cfg.row_density - 1, max(0, int((y_pos / page_height) * cfg.row_density)))
    project = re.sub(r"[^A-Z0-9]", "", meta.get("project", "DOC").upper())
    drawing = re.sub(r"[^A-Z0-9]", "", meta.get("drawing_number", "DWG").upper())
    return f"{project}-{drawing}-{page_no:03d}-{row:03d}-{seq:03d}"


COLUMNS = [
    "sr_no", "praj_document_number", "praj_revision_number", "customer_document_number",
    "customer_revision", "page_sheet", "comment_id", "actual_extracted_comment",
    "snapshot_file", "name_of_person_commented", "date_of_comment", "time_of_comment",
    "comment_color", "is_client_comment", "comment_category", "is_handwritten",
    "extraction_confidence_percent", "annotation_type", "classification_method",
    "visual_feature",
]


def process_pdf(pdf_path: Path, cfg: Config, llm: LLMClient, output_root: Path,
                ocr_engine: Optional[OCREngine] = None, max_pages: Optional[int] = None) -> int:
    job_dir = output_root / pdf_path.stem
    img_dir = job_dir / "images"
    img_dir.mkdir(parents=True, exist_ok=True)

    doc = fitz.open(str(pdf_path))
    try:
        meta = extract_metadata(doc, pdf_path.name, llm, cfg)
        logger.info(f"{pdf_path.name} | metadata: {meta}")

        # ---- collect comments page by page: native annotations first, OCR only if none ----
        comments: List[Dict] = []
        native_pages = ocr_pages = empty_pages = 0
        total_pages = len(doc)
        pages_to_process = min(total_pages, max_pages) if max_pages else total_pages
        logger.info(f"{pdf_path.name} | Processing {pages_to_process} of {total_pages} page(s)")

        for pno in tqdm(range(pages_to_process), desc=f"{pdf_path.stem} pages", leave=False):
            page = doc[pno]
            found = collect_comments(doc, page, pno + 1)          # 1) native annotations
            if found:
                native_pages += 1
            elif ocr_engine is not None and cfg.ocr_enabled:       # 2) nothing found -> OCR
                logger.info(f"Page {pno + 1}: no native annotations -> OCR fallback")
                try:
                    found = ocr_page_to_comments(ocr_engine, page, pno + 1, llm=llm)
                except Exception as e:
                    logger.error(f"Page {pno + 1}: OCR fallback failed: {e}")
                    found = []
                if found:
                    ocr_pages += 1
                else:
                    empty_pages += 1
            else:
                empty_pages += 1
            comments.extend(found)

        logger.info(
            f"{pdf_path.name} | {pages_to_process} pages | {len(comments)} comments | "
            f"{native_pages} native | {ocr_pages} via OCR | {empty_pages} empty"
        )

        # ---- classify ----
        if comments:
            classify_document(comments, llm, cfg)

        # ---- screenshots + optional vision check + records ----
        records: List[Dict] = []
        for n, c in enumerate(tqdm(comments, desc=f"{pdf_path.stem} comments", leave=False), start=1):
            page = doc[c["page"] - 1]
            uid = make_uid(meta, c["page"], c["rect"].y0, page.rect.height, n, cfg)
            shot = capture_screenshot(page, c, uid, img_dir, cfg)
            is_client = "N" if c["author"].lower() in ("praj", "internal", "unknown", "") else "Y"

            is_handwritten = "N"
            category = c["category"]
            confidence = 100
            method = c["method"]
            visual_feature = "NA"

            if cfg.vision_enabled:
                try:
                    v_res = llm.inspect_snapshot_vision(Path(shot), c["text"], c["annot_type"])
                    if v_res and isinstance(v_res, dict):
                        is_handwritten = v_res.get("is_handwritten", "N")
                        visual_feature = v_res.get("visual_feature", "")
                        v_cat = str(v_res.get("category", "")).lower().strip()
                        if v_cat in VALID_CATEGORIES:
                            category = v_cat
                        try:
                            confidence = int(v_res.get("confidence_percent", 100))
                        except (ValueError, TypeError):
                            confidence = 100
                        method = f"{method}+vision"
                except Exception as e:
                    logger.warning(f"Vision check failed for {uid}: {e}")

            records.append({
                "sr_no": n,
                "praj_document_number": meta["drawing_number"],
                "praj_revision_number": meta["praj_revision"],
                "customer_document_number": meta["customer_doc"],
                "customer_revision": meta["customer_revision"],
                "page_sheet": c["page"],
                "comment_id": uid,
                "actual_extracted_comment": c["text"],
                "snapshot_file": shot,
                "name_of_person_commented": c["author"],
                "date_of_comment": c["date"],
                "time_of_comment": c["time"],
                "comment_color": c["color"],
                "is_client_comment": is_client,
                "comment_category": category,
                "is_handwritten": is_handwritten,
                "extraction_confidence_percent": confidence,
                "annotation_type": c["annot_type"],
                "classification_method": method,
                "visual_feature": visual_feature,
            })
    finally:
        doc.close()

    # ---- outputs ----
    with open(job_dir / "comments.json", "w", encoding="utf-8") as f:
        json.dump(records, f, indent=4, ensure_ascii=False)

    df = pd.DataFrame(records, columns=COLUMNS)
    df.to_csv(job_dir / "final_output.csv", index=False)

    total = len(df)
    rows = []
    if total:
        for cat, grp in df.groupby("comment_category"):
            rows.append({"Category": cat, "Count": len(grp),
                         "Percentage": f"{round(len(grp) / total * 100, 1)}%"})
    rows.append({"Category": "TOTAL", "Count": total, "Percentage": "100%" if total else "0%"})
    pd.DataFrame(rows).to_csv(job_dir / "summary.csv", index=False)

    logger.info(f"Done: {pdf_path.stem} | {total} comments | {job_dir}")
    return total

## 5. Run on every PDF in `pdfs/`
Set `MAX_PAGES = 2` for a quick smoke test; `None` processes everything.

In [ ]:
MAX_PAGES = None          # e.g. 2 to test on the first 2 pages of each PDF

cfg = Config.from_yaml(Path("config.yaml"))
llm = LLMClient(cfg)
ocr_engine = OCREngine(cfg) if cfg.ocr_enabled else None   # model loads lazily on the first empty page

pdfs = sorted(PDF_DIR.glob("**/*.pdf"))
if not pdfs:
    raise FileNotFoundError(f"No PDFs in {PDF_DIR}")
logger.info(f"Found {len(pdfs)} PDF(s) | LLM: {cfg.llm_model} | OCR: {cfg.ocr_enabled} ({cfg.ocr_model})")

t0 = time.time()
results: Dict[str, int] = {}
for pdf in tqdm(pdfs, desc="PDFs"):
    try:
        results[pdf.name] = process_pdf(pdf, cfg, llm, OUTPUT_DIR, ocr_engine=ocr_engine, max_pages=MAX_PAGES)
    except Exception as e:
        logger.error(f"Failed on {pdf.name}: {e}", exc_info=True)
        results[pdf.name] = 0

print("=" * 50)
print(f"Completed in {round(time.time() - t0, 1)}s | total comments: {sum(results.values())}")
for name, count in results.items():
    print(f"  {name}: {count}")

## 6. Combine results and download

In [ ]:
import shutil

frames = []
for csv in sorted(OUTPUT_DIR.glob("*/final_output.csv")):
    d = pd.read_csv(csv)
    if len(d):
        d.insert(0, "source_pdf", csv.parent.name + ".pdf")
        frames.append(d)

if frames:
    all_df = pd.concat(frames, ignore_index=True)
    all_df.to_csv(OUTPUT_DIR / "ALL_comments.csv", index=False)
    print(f"{len(all_df)} comments across {len(frames)} PDF(s)")
    display(all_df.drop(columns=["snapshot_file"]).head(30))
    display(all_df.groupby(["source_pdf", "comment_category"]).size().unstack(fill_value=0))
else:
    print("No comments extracted.")

zip_path = shutil.make_archive("/content/comment_extractor_output", "zip", OUTPUT_DIR)
print("Zipped ->", zip_path)

try:
    from google.colab import files
    files.download(zip_path)
except Exception:
    pass